# 2. A difference round trip: sender, database, receiver

Two processes hold the same grid model. One of them changes a few setpoints. What has to travel is the change, not
the model - and it should travel as data, not as a file that has to be parsed again.

This notebook does that round trip inside one process, with two `Network` objects standing in for two clients. The
sender records what it changes and writes it into the database as a new **version**; the receiver walks to that
version and ends up with the sender's state.

In [1]:
import sys, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import pypowsybl as pp
import pypowsybl

from notebook_utils import CGMES_ZIP, NEXT_DAY, SCENARIO, connect, database_url, eq_drift, fresh_scenario, next_day_zip

pp.set_config_read(False)
print(pp.__version__, '->', database_url())

1.17.0.dev1 -> memory:notebook


In [3]:
db = connect()
fresh_scenario(db, SCENARIO)
fresh_scenario(db, NEXT_DAY)
db.load_cgmes(CGMES_ZIP, SCENARIO, '1.0')
db.snapshots(SCENARIO)[['version', 'kind']]

,version,kind
snapshot,,
http://powsybl.org/rdfdb/2021-02-09/snapshot/2021-02-09T19%3A30%3A00Z/1.0,1.0,full


## Sender and receiver start from the same snapshot

In [ ]:
sender = pypowsybl.network.load(CGMES_ZIP)
receiver = pypowsybl.network.load(CGMES_ZIP)

with sender.event_recorder() as recorder:
    sender.update_loads(id=load_id, p0=420.0)
    sender.update_switches(id=switch_id, open=True)
    recorder.to_cgmes_diff("diff.xml")
        # recorder.to_ssh("out.ssh")


# Loads only the load diff, not the full grid model
receiver.update_from_file("diff.xml")

In [5]:
sender = pp.network.from_rdf_db(db, SCENARIO, '1.0')
receiver = pp.network.from_rdf_db(db, SCENARIO, '1.0')

load_id = sorted(sender.get_loads().index)[0]
switch_id = sorted(sender.get_switches().index)[0]
print(load_id, float(sender.get_loads().loc[load_id, 'p0']))

1c6beed6-1acf-42e7-ba55-0cc9f04bddd8 200.0


In [ ]:
sender = pypowsybl.network.from_rdf_db(db, SCENARIO, '1.0')
receiver = pypowsybl.network.from_rdf_db(db, SCENARIO, '1.0')

with sender.event_recorder() as recorder:
    sender.update_loads(id=load_id, p0=420.0)
    sender.update_switches(id=switch_id, open=True)
    recorder.to_rdf_updates(db, SCENARIO, "1.1")


# Loads only the load diff, not the full grid model
receiver.update_from_rdf_db(db, SCENARIO, "1.1")

'diff'

In [ ]:
db.snapshots(SCENARIO)[['version', 'label', 'kind', 'parent', 'depth', 'fast']]

`kind` is `diff` and `fast` is `True`: the change touches only steady-state predicates, so the receiver can apply
it in place instead of rebuilding.

## The receiver walks to it

`update_from_rdf_db` answers with the route it took.

In [ ]:
print(receiver.update_from_rdf_db(db, SCENARIO, '1.1'))
print(float(receiver.get_loads().loc[load_id, 'p0']))

In [ ]:
print(receiver.update_from_rdf_db(db, SCENARIO, '1.1'))   # already there
print(receiver.update_from_rdf_db(db, SCENARIO, '1.0'))   # backwards is a difference too
print(float(receiver.get_loads().loc[load_id, 'p0']))

## The chain of stored models

Every difference says which model it supersedes, which is what makes the chain walkable in both directions.

In [ ]:
db.models(SCENARIO)[['subset', 'kind', 'supersedes', 'fast', 'chain_depth']]

## When a difference cannot be applied: the full route

Not every change can be folded into a network that is already in memory. A difference states properties of CGMES
objects, and the in-place update knows how to set the ones a steady state hypothesis carries - not, for instance,
the *name* of a line. When the path to the target contains such a difference, the loader stops trying and rebuilds
the network from the database. That is the third route, `'full'`, and the caller does not have to ask for it.

`eq_drift(k, label)` is a timestep whose equipment model drifted: the same setpoint change as before, plus one
renamed line.

In [ ]:
db.load_cgmes_from_binary_buffers([eq_drift(2, '21:00')], SCENARIO, '1.2', '21:00')

models = db.models(SCENARIO)
models[models['kind'] == 'diff'][['subset', 'fast', 'supersedes']]

The EQ difference has `fast == False`. Watch what the receiver answers now - and what happens to the Python object:

In [ ]:
before_handle = receiver._handle
print(receiver.update_from_rdf_db(db, SCENARIO, '1.2', '21:00'))
print('same Java network:', receiver._handle is before_handle)
print('variants:', receiver.get_variant_ids())
print([name for name in receiver.get_lines()['name'] if name.startswith('drifted')])

### What a full reload means for the Python object

The object stays valid and keeps its `id`, its per-unit setting and its threading mode - but it now wraps a *new*
Java network. Variants are not carried over, `Network` objects taken from `get_sub_network()` before the reload
still point at the old one, and recorders created before it are refused. The next section shows the refusal.

## Another day, another scenario

A database holds many days. Each is a scenario of its own, with its own root, its own timesteps and its own
versions - and a difference never crosses between them.

In [ ]:
db.load_cgmes_from_binary_buffers([next_day_zip()], NEXT_DAY, '1.0')
db.scenarios()

A sender that belongs to one day may not write its difference into another. The database says so:

In [ ]:
with sender.event_recorder() as recorder:
    sender.update_loads(id=load_id, p0=430.0)
    try:
        recorder.to_rdf_updates(db, NEXT_DAY, '1.1')
    except pp.PyPowsyblError as error:
        print(str(error)[:200])

Walking a network to another day is therefore always a **full reload**, decided without even asking the database:

In [ ]:
print(receiver.update_from_rdf_db(db, NEXT_DAY, '1.0'))
print(receiver.rdf_db_identity()['scenario'])

### Recorders do not survive a reload

A recorder listens to the Java network it was created on. After a full route that network is no longer the one the
Python object wraps, and every export of such a recorder says so rather than quietly writing changes nobody will
see:

In [ ]:
stale = receiver.event_recorder()
stale.start()
print(receiver.update_from_rdf_db(db, SCENARIO, '1.0'))
try:
    stale.to_ssh()
except pp.PyPowsyblError as error:
    print(error)
stale.stop()

In [ ]:
db.close()